# ACHG-CLIP: Experiment B2 (ViT-L/14 Backbone) — miniImageNet (FIXED)

**Bug fixes applied:**
1. ✅ WordNet ID → English class name mapping for CLIP text prompts
2. ✅ Correct HuggingFace model for ViT-L/14 processor
3. ✅ Safe DataLoader num_workers=2 on Linux to prevent worker IPC deadlocks

In [ ]:
import torch
print('='*50)
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name    : {torch.cuda.get_device_name(0)}')
    print(f'Device VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'CUDA Capability: sm_{torch.cuda.get_device_properties(0).major}{torch.cuda.get_device_properties(0).minor}')
print('='*50)

In [ ]:
!pip install -q transformers ftfy regex pyyaml "numpy<2"

import os
if not os.path.exists('/kaggle/working/FSCIL'):
    !git clone https://github.com/Siddarth021/FSCIL.git /kaggle/working/FSCIL
else:
    !git -C /kaggle/working/FSCIL pull

os.chdir('/kaggle/working/FSCIL/ACHG-CLIP')
print(f'Working directory: {os.getcwd()}')

In [ ]:
from data.mini_imagenet_classes import MINI_IMAGENET_CLASS_NAMES, get_mini_imagenet_class_names

print(f'Total classes mapped: {len(MINI_IMAGENET_CLASS_NAMES)}')
print('\nSample mappings:')
for k, v in list(MINI_IMAGENET_CLASS_NAMES.items())[:10]:
    print(f'  {k} -> "{v}"')

sample_prompts = [f'a photo of a {v}' for v in list(MINI_IMAGENET_CLASS_NAMES.values())[:5]]
print('\nSample CLIP prompts:')
for p in sample_prompts:
    print(f'  {p}')

In [ ]:
import os
data_root = '/kaggle/input'
for root, dirs, files in os.walk('/kaggle/input'):
    if 'miniimagenet' in root.lower() or 'mini_imagenet' in root.lower():
        print(f'Found miniImageNet at: {root}')
        data_root = os.path.dirname(root)
        break
        
print(f'Using data_root: {data_root}')

!python run_mini_imagenet.py \
    --variant ViT-L/14 \
    --seed 42 \
    --data_root {data_root}

In [ ]:
import os, json, shutil

results_dir = 'results'
subdirs = [os.path.join(results_dir, d) for d in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, d)) and d.startswith('run_')]

if subdirs:
    latest_run = max(subdirs, key=os.path.getmtime)
    print(f'Latest run folder: {latest_run}')
    
    summary_file = os.path.join(latest_run, 'eval_summary.json')
    if os.path.exists(summary_file):
        with open(summary_file, 'r') as f:
            summary = json.load(f)
        print('\n' + '='*60)
        print('FINAL EVALUATION SUMMARY')
        print('='*60)
        print(json.dumps(summary, indent=2))
        shutil.copy(summary_file, '/kaggle/working/eval_summary.json')

    dest = '/kaggle/working/Backbone_CLIP_C_ViTL14_miniImageNet_fixed'
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(latest_run, dest)
    print(f'\nArtifacts saved to {dest}')